# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [9]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [21]:
# TODO: Load environment variables
from dotenv import load_dotenv
load_dotenv()


True

In [11]:
# TODO: Import the necessary libs
# For example: 
# import os

# from lib.agents import Agent
# from lib.llm import LLM
# from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
# from lib.tooling import tool

# Standard libs
import os
import json
from typing import Any, Dict, List

# Project-provided libraries
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, AIMessage, ToolMessage
from lib.tooling import tool
from lib.agents import Agent   # used for wrapping the state machine

# Vector database
import chromadb

# Tavily
from tavily import TavilyClient

In [23]:
import os

# Make sure the Vocareum key is loaded
from dotenv import load_dotenv
load_dotenv()
print(os.getenv("OPENAI_API_KEY") is not None)  # should print True

# Tell the OpenAI client (used inside lib.llm) to use Vocareum
os.environ["OPENAI_BASE_URL"] = "https://openai.vocareum.com/v1"

from lib.llm import LLM

# Now create the LLM normally (no api_base kwarg)
llm = LLM(model="gpt-4o-mini")

True


In [24]:
# TODO: Load environment variables
# load_dotenv()

# OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
# TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [25]:
# TODO: Create retrieve_game tool
# It should use chroma client and collection you created
# chroma_client = chromadb.PersistentClient(path="chromadb")
# collection = chroma_client.get_collection("udaplay")
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, Playstation 5, Xbox 360...)
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game

from chromadb.utils import embedding_functions

# Use the Vocareum OpenAI endpoint
VOCAREUM_API_KEY = os.getenv("OPENAI_API_KEY") or ""
VOCAREUM_API_BASE = "https://openai.vocareum.com/v1"

# Chroma client for the persistent DB you built in Notebook 1
chroma_client = chromadb.PersistentClient(path="chromadb")

# Embedding function used for query-time embeddings
query_embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=VOCAREUM_API_KEY,
    api_base=VOCAREUM_API_BASE,
)


@tool
def retrieve_game(query: str, top_k: int = 5) -> list[dict[str, Any]]:
    """
    Semantic search: finds the most relevant game entries in the local vector DB.

    Args:
        query:
            A natural language question about video games or the game industry
            (e.g. 'When was God of War Ragnarok released?').
        top_k:
            Maximum number of results to return.

    Returns:
        A list of dictionaries, each containing fields such as:
        - Name:           Game title
        - Platform:       Platform (e.g. Game Boy, PlayStation 5, Xbox 360)
        - YearOfRelease:  Year the game was released on that platform
        - Genre:          Game genre
        - Publisher:      Game publisher
        - Description:    Short description of the game
        - distance:       Vector distance (smaller = more similar)
    """
    # Load (or create, if missing) the same collection as in Notebook 1
    collection = chroma_client.get_or_create_collection(
        name="udaplay",
        embedding_function=query_embedding_fn,
    )

    results = collection.query(
        query_texts=[query],
        n_results=top_k,
        include=["metadatas", "distances"],
    )

    metadatas = results.get("metadatas", [[]])[0]
    distances = results.get("distances", [[]])[0]

    output: list[dict[str, Any]] = []
    for meta, dist in zip(metadatas, distances):
        game_info: dict[str, Any] = {
            "Name": meta.get("Name"),
            "Platform": meta.get("Platform"),
            "YearOfRelease": meta.get("YearOfRelease"),
            "Genre": meta.get("Genre"),
            "Publisher": meta.get("Publisher"),
            "Description": meta.get("Description"),
            "distance": float(dist),
        }
        output.append(game_info)

    return output

#### Evaluate Retrieval Tool

In [27]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result
from pydantic import BaseModel


class EvaluationReport(BaseModel):
    useful: bool
    description: str


def _summarize_retrieved_docs(retrieved_docs: list[dict[str, Any]]) -> str:
    """Format retrieved docs into a compact text block for the judge LLM."""
    if not retrieved_docs:
        return "No documents were retrieved."

    lines: list[str] = []
    for idx, doc in enumerate(retrieved_docs, start=1):
        name = doc.get("Name", "Unknown")
        platform = doc.get("Platform", "Unknown")
        year = doc.get("YearOfRelease", "Unknown")
        desc = doc.get("Description", "") or ""
        # Keep description short to avoid huge prompts
        if len(desc) > 160:
            desc = desc[:160] + "..."
        lines.append(
            f"[{idx}] {name} ({year}) on {platform} – {desc}"
        )
    return "\n".join(lines)


def _extract_json_object(text: str) -> str:
    """
    Helper to robustly pull the JSON object from the model output.
    It tries to find the first '{' and last '}' and keep what's inside.
    """
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        return text
    return text[start : end + 1]


@tool
def evaluate_retrieval(
    question: str,
    retrieved_docs: list[dict[str, Any]],
) -> dict:
    """
    Based on the user's question and on the list of retrieved documents,
    it analyzes whether the documents are useful to answer that question.

    Args:
        question:
            Original question from the user.
        retrieved_docs:
            Retrieved documents most similar to the user query in the Vector
            Database (each item is a dict with Name, Platform, YearOfRelease,
            Description, etc.).

    Returns:
        A dictionary with:
        - useful: whether the documents are useful to answer the question (bool)
        - description: explanation about the evaluation result (str)
    """
    docs_block = _summarize_retrieved_docs(retrieved_docs)

    system_msg = SystemMessage(
        content=(
            "You are an expert evaluator for a game research assistant.\n"
            "Your task is to decide whether the retrieved documents are sufficient "
            "to answer the user's question about games.\n"
            "Always respond ONLY with a JSON object of the form:\n"
            '{\"useful\": <true or false>, \"description\": \"<short explanation>\"}\n'
            "Do NOT include any extra commentary, markdown, or code fences."
        )
    )

    user_msg = UserMessage(
        content=(
            f"User question:\n{question}\n\n"
            f"Retrieved documents:\n{docs_block}"
        )
    )

    # Use the project LLM abstraction (already configured with Vocareum / OpenAI)
    ai_response = llm.invoke([system_msg, user_msg])
    raw_text = getattr(ai_response, "content", str(ai_response)).strip()

    # Try to robustly pull out JSON
    json_candidate = _extract_json_object(raw_text)

    fallback = EvaluationReport(
        useful=False,
        description=f"Could not reliably parse evaluation JSON. Raw model output: {raw_text}",
    )

    try:
        data = json.loads(json_candidate)
        report = EvaluationReport(**data)
        return report.model_dump()
    except Exception:
        return fallback.model_dump()



#### Game Web Search Tool

In [28]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry. 

# Initialize Tavily client once
_tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))


@tool
def game_web_search(question: str, max_results: int = 5) -> dict[str, Any]:
    """
    Web semantic search: uses the Tavily API to look up information about games
    and the game industry on the internet.

    Args:
        question:
            A natural-language question about games or the gaming industry
            (e.g. 'What is Rockstar Games working on right now?').
        max_results:
            Maximum number of search results to retrieve.

    Returns:
        A dictionary with Tavily's response, typically containing:
        - 'answer': an AI-generated summary of the results (if enabled)
        - 'results': a list of individual web pages, each including fields like
            - title
            - url
            - content (short extract or snippet)
    """
    try:
        response = _tavily_client.search(
            query=question,
            max_results=max_results,
            search_depth="advanced",
            include_answer=True,
            include_raw_content=False,
            topic="general",
        )
        # return Tavily's response as-is so the agent can decide how to use it.
        return response

    except Exception as e:
        # In case Tavily fails, return structured error that agent can inspect.
        return {
            "answer": None,
            "results": [],
            "error": f"Tavily search failed: {e}",
        }

### Agent

In [29]:
# TODO: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed
from enum import Enum, auto


class UdaPlayState(Enum):
    """High-level states for the UdaPlay agent workflow."""
    RETRIEVE = auto()
    EVALUATE = auto()
    WEB_SEARCH = auto()
    ANSWER = auto()


class UdaPlayAgent:
    """
    UdaPlay Agent

    Workflow (state machine):

    1. RETRIEVE:
       - Use the internal vector database via `retrieve_game`.

    2. EVALUATE:
       - Ask `evaluate_retrieval` (LLM judge) whether the retrieved docs
         are sufficient to answer the question.

    3. WEB_SEARCH:
       - If internal docs are not enough, call `game_web_search` (Tavily)
         to gather extra information from the web.

    4. ANSWER:
       - Use the main LLM to synthesize a clear, well-cited answer using
         internal docs and (optionally) web search results.
    """

    def __init__(self, llm: LLM):
        self.llm = llm
        # Simple conversation memory: a list of past QA runs
        self.history: list[dict[str, Any]] = []

    # ---------- Helper formatters ----------

    def _format_internal_docs(self, docs: list[dict[str, Any]]) -> str:
        """Turn retrieved game docs into a readable text block for the LLM."""
        if not docs:
            return "No internal documents were retrieved."

        lines: list[str] = []
        for idx, d in enumerate(docs, start=1):
            name = d.get("Name", "Unknown")
            platform = d.get("Platform", "Unknown")
            year = d.get("YearOfRelease", "Unknown")
            genre = d.get("Genre", "Unknown")
            desc = d.get("Description", "") or ""
            if len(desc) > 160:
                desc = desc[:160] + "..."
            lines.append(
                f"[Internal {idx}] {name} ({year}) on {platform} | Genre: {genre}\n"
                f"  {desc}"
            )
        return "\n".join(lines)

    def _format_web_results(self, web_data: dict[str, Any] | None) -> str:
        """Convert Tavily search output into a compact, cited context block."""
        if not web_data:
            return "No web search was performed."

        parts: list[str] = []

        answer = web_data.get("answer")
        if answer:
            parts.append(f"Synthesized web summary:\n{answer}\n")

        results = web_data.get("results", [])
        if results:
            parts.append("Individual web results:")
            for idx, r in enumerate(results, start=1):
                title = r.get("title", "No title")
                url = r.get("url", "No URL")
                parts.append(f"  [Web {idx}] {title} — {url}")

        if not parts:
            return "Web search returned no usable results."

        return "\n".join(parts)

    # ---------- Main run method (state machine) ----------

    def run(self, question: str) -> dict[str, Any]:
        """
        Execute the full agent workflow for a single user question.

        Returns a dict with:
        - 'answer': final natural language answer
        - 'retrieved_docs': internal docs returned by retrieve_game
        - 'evaluation': evaluate_retrieval result
        - 'web_data': Tavily data if web search was used, else None
        """
        state = UdaPlayState.RETRIEVE
        retrieved_docs: list[dict[str, Any]] = []
        evaluation: dict[str, Any] | None = None
        web_data: dict[str, Any] | None = None

        while True:
            # 1) RETRIEVE: use local vector DB
            if state == UdaPlayState.RETRIEVE:
                retrieved_docs = retrieve_game(question)
                state = UdaPlayState.EVALUATE

            # 2) EVALUATE: ask LLM-judge if docs are enough
            elif state == UdaPlayState.EVALUATE:
                evaluation = evaluate_retrieval(
                    question=question,
                    retrieved_docs=retrieved_docs,
                )
                # evaluation is a dict with keys: useful, description
                is_useful = bool(evaluation.get("useful", False))
                state = UdaPlayState.ANSWER if is_useful else UdaPlayState.WEB_SEARCH

            # 3) WEB_SEARCH: fall back to Tavily if needed
            elif state == UdaPlayState.WEB_SEARCH:
                web_data = game_web_search(question)
                state = UdaPlayState.ANSWER

            # 4) ANSWER: synthesize final response
            elif state == UdaPlayState.ANSWER:
                internal_block = self._format_internal_docs(retrieved_docs)
                web_block = self._format_web_results(web_data)
                eval_desc = (
                    evaluation.get("description", "")
                    if isinstance(evaluation, dict)
                    else ""
                )
                eval_useful = (
                    evaluation.get("useful", False)
                    if isinstance(evaluation, dict)
                    else False
                )

                system_msg = SystemMessage(
                    content=(
                        "You are UdaPlay, an AI research assistant for the video game industry.\n"
                        "You have two sources of information:\n"
                        "1) An internal game database (vector store).\n"
                        "2) Web search results.\n\n"
                        "Guidelines:\n"
                        "- Prefer internal data when the evaluation marks it as useful.\n"
                        "- If internal data is incomplete, carefully integrate web results.\n"
                        "- Be honest about uncertainty; never fabricate precise dates/platforms.\n"
                        "- Provide a concise, well-structured answer.\n"
                        "- End with a short 'Sources' section listing either 'Local database'\n"
                        "  and/or the web URLs you used.\n"
                    )
                )

                user_msg = UserMessage(
                    content=(
                        f"User question:\n{question}\n\n"
                        f"Evaluation of internal retrieval:\n"
                        f"  useful={eval_useful}\n"
                        f"  explanation: {eval_desc}\n\n"
                        f"Internal database documents:\n{internal_block}\n\n"
                        f"Web search context (if any):\n{web_block}\n"
                    )
                )

                ai_msg = self.llm.invoke([system_msg, user_msg])
                answer_text = getattr(ai_msg, "content", str(ai_msg))

                result = {
                    "answer": answer_text,
                    "retrieved_docs": retrieved_docs,
                    "evaluation": evaluation,
                    "web_data": web_data,
                }

                # Store trace in memory
                self.history.append(
                    {
                        "question": question,
                        **result,
                    }
                )

                return result


In [31]:
#This is a test to see if the LLM is finally correctly talking to Vocareum.
from lib.messages import SystemMessage, UserMessage

test_msg = llm.invoke([
    SystemMessage(content="You are a test helper."),
    UserMessage(content="Reply with the word: TEST")
])

print(getattr(test_msg, "content", test_msg))

TEST


In [32]:
# TODO: Invoke your agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?
# Instantiate the agent once
uda_agent = UdaPlayAgent(llm=llm)

test_questions = [
    "When were Pokémon Gold and Silver released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for PlayStation 5?",
]

for q in test_questions:
    print("=" * 80)
    print("QUESTION:", q)
    result = uda_agent.run(q)

    print("\nANSWER:\n")
    print(result["answer"])

    print("\n--- Debug info ---")
    print("Evaluation:", result["evaluation"])
    print("Num internal docs:", len(result["retrieved_docs"]))
    print("Web search used:", result["web_data"] is not None)
    print()

QUESTION: When were Pokémon Gold and Silver released?

ANSWER:

Pokémon Gold and Silver were released in Japan on November 21, 1999, and in North America on October 15, 2000. These games were the first in the Pokémon series designed specifically for the Game Boy Color.

Sources:
- Web 1: https://www.pokemon.com/us/pokemon-video-games/pokemon-gold-version-and-pokemon-silver-version
- Web 3: https://en.wikipedia.org/wiki/Pok%C3%A9mon_Gold_and_Silver

--- Debug info ---
Evaluation: {'useful': False, 'description': 'The retrieved documents do not provide the release date for Pokémon Gold and Silver.'}
Num internal docs: 5
Web search used: True

QUESTION: Which one was the first 3D platformer Mario game?

ANSWER:

The first 3D platformer Mario game is **Super Mario 64**, released in 1996 for the Nintendo 64. This game was groundbreaking for its time, setting new standards for the platforming genre and featuring Mario's quest to rescue Princess Peach.

Sources:
- Local database

--- Debug in

### (Optional) Advanced

In [ ]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes